#Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

# Load Dataset

In [ ]:
folder_path = '/content/drive/MyDrive/PIZZA_CARBONARA_shared_folder/LLM/'

In [ ]:
SEED = 82
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
df = pd.read_csv(folder_path + "data/train_enriched.csv")
df_val = pd.read_csv(folder_path + "data/val_enriched.csv")

In [ ]:
df.head()

,Unnamed: 0,item,name,description,type,category,subcategory,label,origin_lang
0,0,http://www.wikidata.org/entity/Q32786,916,2012 film by M. Mohanan,entity,films,film,cultural exclusive,ml
1,1,http://www.wikidata.org/entity/Q371,!!!,American dance-punk band from California,entity,music,musical group,cultural representative,en
2,2,http://www.wikidata.org/entity/Q3729947,¡Soborno!,Mort & Phil comic,entity,comics and anime,comics,cultural representative,es
3,3,http://www.wikidata.org/entity/Q158611,+44,American band,entity,music,musical group,cultural representative,en
4,4,http://www.wikidata.org/entity/Q280375,1 Monk Street,"building in Monmouth, Wales",entity,architecture,building,cultural exclusive,en


# Create input

In [ ]:
items = df['item']
names = df['name']
descriptions = df['description']
types = df['type']
categories = df['category']
subcategories = df['subcategory']
labels = df['label']
language = df["origin_lang"]

In [ ]:
def create_prompt(row):
    lang_tag = row['origin_lang']
    return (
        f"You are a classification expert. "
        f"Based on the information below, classify the item into the correct category:\n"
        f"[LANG] {lang_tag.upper()} "
        f"[NAME] {row['name']} "
        f"[DESC] {row['description']} "
        f"[TYPE] {row['type']} "
    )

In [ ]:
df['text'] = df.apply(lambda row: create_prompt(row), axis=1)
df_val['text'] = df_val.apply(lambda row: create_prompt(row), axis=1)

In [ ]:
df.head()

,Unnamed: 0,item,name,description,type,category,subcategory,label,origin_lang,text
0,0,http://www.wikidata.org/entity/Q32786,916,2012 film by M. Mohanan,entity,films,film,cultural exclusive,ml,You are a classification expert. Based on the ...
1,1,http://www.wikidata.org/entity/Q371,!!!,American dance-punk band from California,entity,music,musical group,cultural representative,en,You are a classification expert. Based on the ...
2,2,http://www.wikidata.org/entity/Q3729947,¡Soborno!,Mort & Phil comic,entity,comics and anime,comics,cultural representative,es,You are a classification expert. Based on the ...
3,3,http://www.wikidata.org/entity/Q158611,+44,American band,entity,music,musical group,cultural representative,en,You are a classification expert. Based on the ...
4,4,http://www.wikidata.org/entity/Q280375,1 Monk Street,"building in Monmouth, Wales",entity,architecture,building,cultural exclusive,en,You are a classification expert. Based on the ...


## Final input

In [ ]:
print(df.iloc[0]["text"])
print()
print(df.iloc[2000]["text"])

You are a classification expert. Based on the information below, classify the item into the correct category:
[LANG] ML [NAME] 916 [DESC] 2012 film by M. Mohanan [TYPE] entity 

You are a classification expert. Based on the information below, classify the item into the correct category:
[LANG] PT [NAME] Fernanda Brandão [DESC] Brazilian singer and dancer [TYPE] entity 


# Model

In [ ]:
special_tokens = ['[LANG]','[NAME]', '[DESC]', '[TYPE]']

In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

4

In [ ]:
label_map = {
    'cultural agnostic': 0,
    'cultural exclusive': 1,
    'cultural representative': 2
}

df['label'] = df['label'].map(label_map).tolist()
df_val['label'] = df_val['label'].map(label_map).tolist()

In [ ]:
df.head()

,item,name,description,type,category,subcategory,label,origin_lang,text
0,http://www.wikidata.org/entity/Q32786,916,2012 film by M. Mohanan,entity,films,film,1,ml,You are a classification expert. Based on the ...
1,http://www.wikidata.org/entity/Q371,!!!,American dance-punk band from California,entity,music,musical group,2,en,You are a classification expert. Based on the ...
2,http://www.wikidata.org/entity/Q3729947,¡Soborno!,Mort & Phil comic,entity,comics and anime,comics,2,es,You are a classification expert. Based on the ...
3,http://www.wikidata.org/entity/Q158611,+44,American band,entity,music,musical group,2,en,You are a classification expert. Based on the ...
4,http://www.wikidata.org/entity/Q280375,1 Monk Street,"building in Monmouth, Wales",entity,architecture,building,1,en,You are a classification expert. Based on the ...


In [ ]:
model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250006, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

## Peparate data for training

In [ ]:
# Dataset
class PromptDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'label': self.labels[idx]
        }

In [ ]:
def collate_fn(batch):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256)
    enc["labels"] = labels
    return enc

In [ ]:
labels = df["label"].tolist()
texts = df["text"].tolist()

In [ ]:
labels_val = df_val["label"].tolist()
texts_val = df_val["text"].tolist()

In [ ]:
BATCH_SIZE = 16

In [ ]:
train_loader = DataLoader(
    PromptDataset(texts, labels),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    PromptDataset(texts_val, labels_val),
    batch_size=2 * BATCH_SIZE,
    collate_fn=collate_fn,
)

# R-Drop Loss

In [ ]:
def rdrop_loss(logits_a, logits_b, targets, alpha: float = 0.5, eps: float = 1e-6):
    """Cross‑entropy + symmetric KL with clamped probabilities (NaN‑safe)."""

    ce = (
        F.cross_entropy(logits_a, targets, label_smoothing=0.1)
        + F.cross_entropy(logits_b, targets, label_smoothing=0.1)
    ) / 2

    log_p = F.log_softmax(logits_a, dim=-1)
    log_q = F.log_softmax(logits_b, dim=-1)

    # Clamp probabilities to avoid zeros → −inf products
    p = torch.exp(log_p).clamp(min=eps)
    q = torch.exp(log_q).clamp(min=eps)

    # Renormalise to ensure ∑p = ∑q = 1 after clamping
    p = p / p.sum(dim=-1, keepdim=True)
    q = q / q.sum(dim=-1, keepdim=True)

    kl1 = F.kl_div(log_p, q, reduction="batchmean", log_target=False)
    kl2 = F.kl_div(log_q, p, reduction="batchmean", log_target=False)

    return ce + alpha * (kl1 + kl2) / 2

# Optimazer

In [ ]:
EPOCHS = 10
LR = 2e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_total = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * steps_total), num_training_steps=steps_total
)

CLIP_NORM = 0.9  # gradient clipping

# Training

In [ ]:
best_val_acc = 0.0
CKPT_PATH = Path("xlmr_rdrop_best.pt")

In [ ]:
for epoch in range(1, EPOCHS + 1):
    # ---------------- Train ----------------
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]"):
        batch = {k: v.to(device) for k, v in batch.items()}
        out1 = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        out2 = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])

        loss = rdrop_loss(out1.logits, out2.logits, batch["labels"])
        if torch.isnan(loss):
            optimizer.zero_grad(set_to_none=True)
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        train_loss += loss.item()
        preds = torch.argmax(out1.logits, -1)
        train_correct += (preds == batch["labels"]).sum().item()
        train_total += preds.size(0)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # ---------------- Validation ----------------
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]"):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            loss = F.cross_entropy(logits, batch["labels"])  # standard CE for val
            val_loss += loss.item()
            preds = torch.argmax(logits, -1)
            val_correct += (preds == batch["labels"]).sum().item()
            val_total += preds.size(0)
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    print(
        f"\n📝 Epoch {epoch}/{EPOCHS} → "
        f"train_loss: {train_loss:.4f} · train_acc: {train_acc:.4f} | "
        f"val_loss: {val_loss:.4f} · val_acc: {val_acc:.4f}"
    )

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"📌  New best model saved @ val_acc = {val_acc:.4f}\n")

Epoch 1/10 [val]: 100%|██████████| 10/10 [00:01<00:00,  7.61it/s]



📝 Epoch 1/10 → train_loss: 0.8736 · train_acc: 0.6305 | val_loss: 0.6405 · val_acc: 0.7033
📌  New best model saved @ val_acc = 0.7033



Epoch 2/10 [val]: 100%|██████████| 10/10 [00:01<00:00,  7.59it/s]



📝 Epoch 2/10 → train_loss: 0.6819 · train_acc: 0.7775 | val_loss: 0.5993 · val_acc: 0.7533
📌  New best model saved @ val_acc = 0.7533



Epoch 3/10 [val]: 100%|██████████| 10/10 [00:01<00:00,  7.54it/s]


...


📝 Epoch 8/10 → train_loss: 0.3516 · train_acc: 0.9779 | val_loss: 0.7828 · val_acc: 0.7733


Epoch 9/10 [val]: 100%|██████████| 10/10 [00:01<00:00,  7.61it/s]



📝 Epoch 9/10 → train_loss: 0.3358 · train_acc: 0.9845 | val_loss: 0.8223 · val_acc: 0.7600


Epoch 10/10 [val]: 100%|██████████| 10/10 [00:01<00:00,  7.56it/s]


📝 Epoch 10/10 → train_loss: 0.3244 · train_acc: 0.9894 | val_loss: 0.8239 · val_acc: 0.7667


# Final Evaluation

In [ ]:
print("\n===== Final validation performance (best checkpoint) =====")
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        all_preds.extend(torch.argmax(logits, -1).cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

print("Validation accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds, zero_division=0))


===== Final validation performance (best checkpoint) =====


/tmp/ipykernel_31/3434250934.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(CKPT_PATH, map_location=device))


Validation accuracy: 0.7966666666666666
              precision    recall  f1-score   support

           0       0.89      0.95      0.92       117
           1       0.75      0.62      0.68        76
           2       0.72      0.76      0.74       107

    accuracy                           0.80       300
   macro avg       0.79      0.77      0.78       300
weighted avg       0.79      0.80      0.79       300



In [ ]:
print(f"\nModel weights saved to {CKPT_PATH.resolve()}")


Model weights saved to /kaggle/working/xlmr_rdrop_best.pt
